# Etapa 3 — Matriz integrada + Clustering (Pipeline principal)

Ensambla la **Hospital_Matrix_Integrada** principal y aplica el clustering. La matriz combina:

- 6 tradicionales + 2 diversidad + 2 CMA
- 5–8 componentes PCA de la casuística clínica (CIE-10 + CIE-9-MC + Top-20)
- 18 features extendidas/agregadas (demográficas, ingreso, alta, procedencia, pabellón, obstétricas, estancia)

**Técnica principal: Aglomerativo Ward** (jerárquico), con K-means y GMM como contraste. La dotación de camas queda excluida.

In [1]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert np.__version__.startswith('1.'), 'NumPy debe ser 1.x'

from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
    davies_bouldin_score)
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import RobustScaler

from src.modeling.matriz import Constructor_Matriz
from src.modeling.reducer import Reductor_Dimensional
from src.utils.io import read_parquet, write_parquet, TABLES_DIR, FIGURES_DIR, PROCESSED_DIR

RANDOM_STATE = 42
sns.set_theme(style='whitegrid', font_scale=1.05)
pd.set_option('display.float_format', '{:.4f}'.format)

## 3.1 Carga de features y casuística

In [2]:
f_trad = read_parquet('features_tradicionales')
f_div = read_parquet('features_diversidad')
f_cma = read_parquet('features_cma')
f_ext = read_parquet('features_extendidas')
v_caps = read_parquet('casuistica_capitulos')
v_secs = read_parquet('casuistica_procedimientos')
v_top20 = read_parquet('casuistica_top20_grds')

v_cap_principal = v_caps[v_caps['variante']=='principal'].drop(columns='variante').reset_index(drop=True)
v_cap_ponderado = v_caps[v_caps['variante']=='ponderado'].drop(columns='variante').reset_index(drop=True)
print('trad', f_trad.shape, '| div', f_div.shape, '| cma', f_cma.shape, '| ext', f_ext.shape)

trad (65, 7) | div (65, 3) | cma (65, 3) | ext (65, 19)


## 3.2 Reducción dimensional PCA (selección de variante de capítulos)

In [3]:
def concatenar(v_cap):
    return (v_cap.merge(v_secs, on='COD_HOSPITAL', how='inner', suffixes=('', '_sec'))
                 .merge(v_top20, on='COD_HOSPITAL', how='inner', suffixes=('', '_top')))

resultados = {}
for nombre, v_cap in [('principal', v_cap_principal), ('ponderado', v_cap_ponderado)]:
    cas = concatenar(v_cap)
    red = Reductor_Dimensional(matriz_casuistica=cas)
    pca_res = red.fit_pca()
    # matriz candidata para evaluar variante: trad+div+cma+pca+ext
    cand = Constructor_Matriz(
        features_tradicionales=f_trad, features_diversidad=f_div,
        features_cma=f_cma, casuistica_reducida=pca_res['componentes'],
        features_extendidas=f_ext,
    ).construir()
    Xc = cand.drop(columns=['COD_HOSPITAL','peso_medio_cma','peso_medio_cma_imputado'])
    for c in ['egresos_por_anio','edad_mediana','pabellones_promedio']:
        if c in Xc: Xc[c] = np.log1p(Xc[c].clip(lower=0))
    Xs = RobustScaler().fit_transform(Xc.fillna(0.0).values)
    Zc = linkage(Xs, method='ward')
    sils = {k: silhouette_score(Xs, fcluster(Zc, t=k, criterion='maxclust')) for k in (2,3,4,5)}
    best = max(sils.values())
    resultados[nombre] = {'pca': pca_res, 'sil_max': best, 'sils': sils}
    print(f'{nombre}: K_pca={pca_res["K"]} var={pca_res["explained_variance_ratio"].sum():.3f} '
          f'sil_max(K2-5)={best:.3f}')

VARIANTE = max(resultados, key=lambda k: resultados[k]['sil_max'])
res_win = resultados[VARIANTE]['pca']
print(f'\n>>> Variante ganadora: {VARIANTE}')

principal: K_pca=8 var=0.717 sil_max(K2-5)=0.659
ponderado: K_pca=8 var=0.725 sil_max(K2-5)=0.663

>>> Variante ganadora: ponderado


## 3.3 Ensamblaje de la Hospital_Matrix_Integrada principal

In [4]:
write_parquet(res_win['componentes'], 'casuistica_reducida',
    expected_cols=['COD_HOSPITAL'] + [f'dim_{i+1:02d}' for i in range(res_win['K'])])
res_win['loadings'].to_csv(TABLES_DIR / 'reduccion_dimensional.csv')

constructor_m = Constructor_Matriz(
    features_tradicionales=f_trad, features_diversidad=f_div,
    features_cma=f_cma, casuistica_reducida=res_win['componentes'],
    features_extendidas=f_ext,
)
matriz = constructor_m.construir()
constructor_m.construir_y_persistir()
print('Hospital_Matrix_Integrada:', matriz.shape)
n_feats = matriz.shape[1] - 3  # menos COD_HOSPITAL, peso_medio_cma_imputado y... 
print('Features para clustering:', matriz.drop(columns=['COD_HOSPITAL','peso_medio_cma','peso_medio_cma_imputado']).shape[1])

Hospital_Matrix_Integrada: (65, 38)
Features para clustering: 35


## 3.4 Preprocesamiento (log1p + RobustScaler)

In [5]:
COLS_DROP = ['COD_HOSPITAL', 'peso_medio_cma', 'peso_medio_cma_imputado']
feats = matriz.drop(columns=[c for c in COLS_DROP if c in matriz.columns]).copy()
feats.index = matriz['COD_HOSPITAL'].astype(str).values
feats = feats.fillna(0.0)
LOG1P = ['egresos_por_anio', 'edad_mediana', 'pabellones_promedio']
for c in LOG1P:
    if c in feats.columns:
        feats[c] = np.log1p(feats[c].clip(lower=0))
X = RobustScaler().fit_transform(feats.values)
ids = feats.index.tolist()
print('X:', X.shape)

X: (65, 35)


## 3.5 Clustering — Aglomerativo Ward (principal) + K-means + GMM

Se evalúan tres técnicas en K=2..10. La técnica principal de la tesis es **Aglomerativo Ward** por su estabilidad con n pequeño y su jerarquía interpretable (dendrograma).

In [6]:
K_GRID = list(range(2, 11))
Z = linkage(X, method='ward')
filas = []
asig_ward = {}
for k in K_GRID:
    lab = fcluster(Z, t=k, criterion='maxclust') - 1
    asig_ward[k] = lab
    filas.append({'metodo':'Aglomerativo','K':k,
        'silhouette':silhouette_score(X,lab),
        'calinski_harabasz':calinski_harabasz_score(X,lab),
        'davies_bouldin':davies_bouldin_score(X,lab),
        'tamanos':str(sorted(pd.Series(lab).value_counts().tolist(), reverse=True))})
for k in K_GRID:
    km = KMeans(n_clusters=k, n_init=50, random_state=RANDOM_STATE).fit(X)
    filas.append({'metodo':'K-means','K':k,'silhouette':silhouette_score(X,km.labels_),
        'calinski_harabasz':calinski_harabasz_score(X,km.labels_),
        'davies_bouldin':davies_bouldin_score(X,km.labels_),
        'tamanos':str(sorted(pd.Series(km.labels_).value_counts().tolist(), reverse=True))})
for k in K_GRID:
    gm = GaussianMixture(n_components=k, n_init=10, random_state=RANDOM_STATE, max_iter=200).fit(X)
    lab = gm.predict(X)
    if len(set(lab)) < 2: continue
    filas.append({'metodo':'GMM','K':k,'silhouette':silhouette_score(X,lab),
        'calinski_harabasz':calinski_harabasz_score(X,lab),
        'davies_bouldin':davies_bouldin_score(X,lab),
        'tamanos':str(sorted(pd.Series(lab).value_counts().tolist(), reverse=True))})
df_met = pd.DataFrame(filas)
df_met.to_csv(TABLES_DIR / 'clustering_metricas_k.csv', index=False)
print(df_met[df_met['metodo']=='Aglomerativo'][['K','silhouette','davies_bouldin','tamanos']].to_string(index=False))

 K  silhouette  davies_bouldin                           tamanos
 2      0.6629          0.4230                           [62, 3]
 3      0.4363          1.3325                        [53, 9, 3]
 4      0.4140          1.0222                     [53, 7, 3, 2]
 5      0.2033          1.4164                 [39, 14, 7, 3, 2]
 6      0.2157          1.3606              [39, 14, 4, 3, 3, 2]
 7      0.2146          1.1370           [39, 14, 4, 3, 3, 1, 1]
 8      0.2182          0.9602        [39, 14, 4, 3, 2, 1, 1, 1]
 9      0.1135          1.0866    [23, 16, 14, 4, 3, 2, 1, 1, 1]
10      0.1217          1.0071 [23, 16, 13, 4, 3, 2, 1, 1, 1, 1]


## 3.6 Selección de K (Aglomerativo)

In [7]:
ward = df_met[df_met['metodo']=='Aglomerativo'].copy()
# K=2 es el corte natural mas fuerte pero trivial (institutos vs resto).
# La particion de PERFILAMIENTO principal de la tesis es K=4: separa los
# 3 grupos especiales (pediatricos, institutos, alta-CMA) del mainstream,
# y luego el mainstream se subdivide en la etapa 4 (clustering jerarquico
# de 2 niveles). Se prefiere K=4 sobre K=3 por interpretabilidad: K=4
# separa institutos de alta-CMA con Silhouette casi identico.
K_NATURAL = int(ward.loc[ward['silhouette'].idxmax(), 'K'])
K_OPTIMO = 4  # particion de perfilamiento principal (nivel 1)
sil_k4 = float(ward.loc[ward['K']==K_OPTIMO, 'silhouette'].iloc[0])
print(f'K corte natural (max Silhouette global): {K_NATURAL} '
      f"(Sil {ward['silhouette'].max():.3f})")
print(f'>>> K_OPTIMO (perfilamiento nivel 1) = {K_OPTIMO} (Sil {sil_k4:.3f})')

K corte natural (max Silhouette global): 2 (Sil 0.663)
>>> K_OPTIMO (perfilamiento nivel 1) = 4 (Sil 0.414)


## 3.7 Bootstrap del Silhouette (100 iteraciones)

In [8]:
rng = np.random.default_rng(RANDOM_STATE)
n = X.shape[0]; n_sub = max(2, int(n*0.8))
sils = []
for _ in range(100):
    idx = rng.choice(n, size=n_sub, replace=False)
    Zb = linkage(X[idx], method='ward')
    lab = fcluster(Zb, t=K_OPTIMO, criterion='maxclust')
    if len(set(lab)) > 1:
        sils.append(silhouette_score(X[idx], lab))
sils = np.array(sils)
boot = {'silhouette_media':float(sils.mean()), 'silhouette_std':float(sils.std(ddof=1)),
        'silhouette_ic_inf':float(np.percentile(sils,2.5)), 'silhouette_ic_sup':float(np.percentile(sils,97.5))}
pd.DataFrame([{'K_optimo':K_OPTIMO, **boot}]).to_csv(TABLES_DIR / 'clustering_bootstrap.csv', index=False)
print(f"Silhouette bootstrap K={K_OPTIMO}: media={boot['silhouette_media']:.3f} "
      f"IC95%=[{boot['silhouette_ic_inf']:.3f}, {boot['silhouette_ic_sup']:.3f}]")

Silhouette bootstrap K=4: media=0.408 IC95%=[0.242, 0.504]


## 3.8 Ajuste definitivo + persistencia de asignaciones

In [9]:
labels = asig_ward[K_OPTIMO]
asign = pd.DataFrame({'COD_HOSPITAL': ids, 'cluster': labels.astype('int8')})
write_parquet(asign, 'hospital_clusters_integrado', expected_cols=['COD_HOSPITAL','cluster'])
asign.to_csv(TABLES_DIR / 'asignacion_clusters_final.csv', index=False)
print('Distribución K_OPTIMO:')
print(asign['cluster'].value_counts().sort_index().to_string())

Distribución K_OPTIMO:
cluster
0     3
1    53
2     2
3     7


## 3.9 Dendrograma + PCA 2D

In [10]:
fig, ax = plt.subplots(figsize=(15, 6))
dendrogram(Z, labels=ids, leaf_rotation=90, leaf_font_size=6,
           color_threshold=Z[-K_OPTIMO+1, 2], above_threshold_color='gray')
ax.set_title(f'Dendrograma Aglomerativo Ward — Hospital_Matrix_Integrada (n={len(ids)})')
ax.set_ylabel('Distancia (Ward)')
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'dendrograma_ward.png', dpi=150, bbox_inches='tight'); plt.close()

pca2d = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca2d.fit_transform(X)
fig, ax = plt.subplots(figsize=(10, 7))
for c in sorted(set(labels)):
    m = labels == c
    ax.scatter(coords[m,0], coords[m,1], s=80, edgecolors='black', linewidth=0.7, label=f'C{c} (n={m.sum()})')
ax.set_xlabel(f'PC1 ({pca2d.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca2d.explained_variance_ratio_[1]:.1%})')
ax.set_title(f'PCA 2D — Aglomerativo Ward K={K_OPTIMO}'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'clustering_pca2d_integrado.png', dpi=150, bbox_inches='tight'); plt.close()
print('Figuras: dendrograma_ward.png, clustering_pca2d_integrado.png')

Figuras: dendrograma_ward.png, clustering_pca2d_integrado.png


## 3.10 Estado del pipeline + resumen

In [11]:
state = {
    'tecnica_principal': 'Aglomerativo Ward',
    'variante_capitulos': VARIANTE,
    'K_pca': int(res_win['K']),
    'varianza_pca': float(res_win['explained_variance_ratio'].sum()),
    'K_natural': int(K_NATURAL),
    'K_optimo': int(K_OPTIMO),
    'silhouette_K_optimo': float(ward.loc[ward['K']==K_OPTIMO, 'silhouette'].iloc[0]),
    'silhouette_K_natural': float(ward.loc[ward['K']==K_NATURAL, 'silhouette'].iloc[0]),
    'silhouette_media_bootstrap': boot['silhouette_media'],
    'silhouette_ic_inf': boot['silhouette_ic_inf'],
    'silhouette_ic_sup': boot['silhouette_ic_sup'],
    'n_features': int(feats.shape[1]),
    'n_hospitales': int(len(ids)),
}
(PROCESSED_DIR / 'pipeline_state.json').write_text(json.dumps(state, indent=2))
print('=' * 60)
print('RESUMEN ETAPA 3 — Clustering principal')
print('=' * 60)
for k, v in state.items():
    print(f'  {k}: {v}')
print('\nSiguiente: 04_xai_outliers.ipynb')

RESUMEN ETAPA 3 — Clustering principal
  tecnica_principal: Aglomerativo Ward
  variante_capitulos: ponderado
  K_pca: 8
  varianza_pca: 0.7252257944492694
  K_natural: 2
  K_optimo: 4
  silhouette_K_optimo: 0.41396518156048984
  silhouette_K_natural: 0.6629213838661951
  silhouette_media_bootstrap: 0.4075667605059952
  silhouette_ic_inf: 0.24172809394041073
  silhouette_ic_sup: 0.5041522337985953
  n_features: 35
  n_hospitales: 65

Siguiente: 04_xai_outliers.ipynb
